# Capítulo 8 — Robustez Adversarial em IA para Defesa

**Inteligência Artificial e Machine Learning Avançado para Defesa** — Prof. Cristiano Alves · Quantum Strategic AI

---

### 🎯 Objetivos do capítulo

Ao final deste notebook, você será capaz de:

1. **Compreender** por que modelos de aprendizado são **intrinsecamente vulneráveis** à manipulação e por que isso é singularmente grave em defesa, onde o adversário é real e adaptativo;
2. **Distinguir** as principais classes de ataque — **envenenamento de dados** (no treino), **exemplos adversariais** (na inferência), ataques físicos e extração de modelos;
3. **Compreender e implementar** exemplos adversariais — **FGSM** e **PGD** — e os modelos de ameaça de caixa-branca e caixa-preta;
4. **Aplicar** defesas algorítmicas — **treinamento adversarial**, destilação defensiva, detecção — e reconhecer seus **limites**: nenhuma defesa é completa;
5. **Aplicar** a **verificação e validação (V&V)** de modelos em sistemas críticos — a disciplina de estabelecer confiança justificada antes do emprego —, que fecha o Módulo III e prepara o Módulo IV.

> Os três primeiros módulos construíram modelos que percebem e agem. Todos partilham um pressuposto silencioso, herdado do aprendizado estatístico: o de que **os dados de emprego se assemelham aos de treino**. Em defesa, esse pressuposto é **falso por construção**. Aqui há um **adversário** — real, inteligente e adaptativo — cujo propósito é justamente fazer o modelo falhar. Ele não é um ruído aleatório a tolerar; é um oponente que **estuda o sistema, encontra suas brechas e as explora deliberadamente**.
>
> Uma verdade incômoda atravessa o capítulo: os modelos de aprendizado são **surpreendentemente frágeis**, e a acurácia de laboratório nada diz sobre a resistência a um oponente que tenta derrotá-los. Reconhecer essa fragilidade com honestidade não é pessimismo — é, ele próprio, **uma postura de segurança**.

## Preparação do ambiente

Todo o capítulo executa com `numpy`, `matplotlib` e `torch`, já presentes no Colab. Treinamos um pequeno classificador de imagens **do zero** — sobre um conjunto sintético de "sinais de sensor" gerado no próprio notebook — e o submetemos a ataques e defesas. Nada de *download* de pesos; tudo roda em CPU em poucos minutos (uma GPU acelera o treinamento adversarial da Seção 8.4).

Como sempre, fixamos a **semente aleatória**: *dados os mesmos dados, os mesmos resultados*.

In [ ]:
# Preparacao do ambiente: importacoes e sementes de reprodutibilidade
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEMENTE = 0
np.random.seed(SEMENTE)
torch.manual_seed(SEMENTE)
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"

print(f"numpy      {np.__version__}")
print(f"pytorch    {torch.__version__}")
print(f"dispositivo: {DISPOSITIVO}")
print("Ambiente pronto.")

---
Antes da teoria, precisamos de um **modelo-alvo**: um classificador que atinja alta acurácia "de laboratório", para depois vermos essa acurácia **desabar** sob ataque. Geramos um conjunto sintético de **manchas de sensor** $16 \times 16$ em quatro classes — cada classe é uma assinatura visual distinta (uma barra vertical, uma horizontal, um ponto, um anel) sobre fundo ruidoso — e treinamos sobre ele uma CNN compacta, no espírito do Capítulo 3.

In [ ]:
# Conjunto sintetico de "assinaturas de sensor": 4 classes 16x16
LADO, N_CLASSES = 16, 4

def gera_dados(n_por_classe=800, semente=SEMENTE):
    rng = np.random.default_rng(semente)
    X, y = [], []
    for classe in range(N_CLASSES):
        for _ in range(n_por_classe):
            img = rng.normal(0.15, 0.05, (LADO, LADO))
            cx, cy = rng.integers(4, 12, 2)
            if classe == 0:                     # barra vertical
                img[cy - 3:cy + 3, cx] += 0.8
            elif classe == 1:                   # barra horizontal
                img[cy, cx - 3:cx + 3] += 0.8
            elif classe == 2:                   # ponto brilhante
                img[cy - 1:cy + 1, cx - 1:cx + 1] += 0.9
            else:                               # anel
                yy, xx = np.ogrid[:LADO, :LADO]
                anel = np.abs((xx - cx) ** 2 + (yy - cy) ** 2 - 6) < 3
                img[anel] += 0.7
            X.append(np.clip(img, 0, 1))
            y.append(classe)
    X = np.array(X, dtype=np.float32)[:, None, :, :]   # (n, 1, 16, 16)
    y = np.array(y, dtype=np.int64)
    ordem = rng.permutation(len(X))
    return X[ordem], y[ordem]

X, y = gera_dados()
n_tr = int(0.8 * len(X))
X_tr, y_tr = X[:n_tr], y[:n_tr]
X_te, y_te = X[n_tr:], y[n_tr:]
print(f"treino: {X_tr.shape} | teste: {X_te.shape} | {N_CLASSES} classes")

NOMES = ["barra |", "barra --", "ponto", "anel O"]
fig, eixos = plt.subplots(1, 4, figsize=(9, 2.6))
for c in range(N_CLASSES):
    eixos[c].imshow(X[y == c][0, 0], cmap="gray")
    eixos[c].set_title(NOMES[c]); eixos[c].axis("off")
plt.suptitle("As quatro assinaturas de sensor (sinteticas)")
plt.tight_layout(); plt.show()

In [ ]:
# Uma CNN compacta (Cap. 3), treinada ate alta acuracia "de laboratorio"
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, 64), nn.ReLU(),
            nn.Linear(64, N_CLASSES),
        )

    def forward(self, x):
        return self.rede(x)

def treina(modelo, X_tr, y_tr, epocas=8, lote=128, semente=SEMENTE):
    modelo.to(DISPOSITIVO).train()
    otim = torch.optim.Adam(modelo.parameters(), lr=1e-3)
    Xt = torch.tensor(X_tr, device=DISPOSITIVO)
    yt = torch.tensor(y_tr, device=DISPOSITIVO)
    rng = np.random.default_rng(semente)
    for _ in range(epocas):
        for i in rng.permutation(len(Xt))[:len(Xt) // lote * lote].reshape(-1, lote):
            idx = torch.tensor(i, device=DISPOSITIVO)
            perda = F.cross_entropy(modelo(Xt[idx]), yt[idx])
            otim.zero_grad(); perda.backward(); otim.step()
    return modelo

def acuracia(modelo, X, y):
    modelo.eval()
    with torch.no_grad():
        pred = modelo(torch.tensor(X, device=DISPOSITIVO)).argmax(1)
    return (pred.cpu().numpy() == y).mean()

torch.manual_seed(SEMENTE)
modelo = treina(CNN(), X_tr, y_tr)
print(f"acuracia LIMPA no teste: {acuracia(modelo, X_te, y_te):.3f}")
print("Um modelo de laboratorio impecavel -- guarde este numero.")

**Observe:** um classificador quase perfeito no conjunto de teste. É esse o número que a acurácia de laboratório reporta — e é exatamente essa confiança que o restante do capítulo vai **desmontar**.

## 8.1 Por que modelos de aprendizado são vulneráveis

A vulnerabilidade **não é um defeito de implementação** a corrigir, mas uma consequência do modo como os modelos aprendem. Um modelo estatístico captura **correlações** que separam bem as classes nos dados de treino, não **conceitos robustos**. Em espaços de alta dimensão, as fronteiras de decisão passam **surpreendentemente perto** de exemplos corretamente classificados — de modo que uma perturbação minúscula, imperceptível a um humano, pode empurrar uma entrada para o outro lado da fronteira e inverter a predição.

A isso soma-se a **quebra deliberada do pressuposto estatístico**. O aprendizado supervisionado assume que treino e teste vêm da mesma distribuição; o adversário existe precisamente para deslocá-la. Convém organizar os ataques por três eixos:

- **Momento** — ataques no *treino* (**envenenamento**) corrompem o modelo antes que ele exista; ataques na *inferência* (**evasão**) manipulam a entrada de um modelo já treinado;
- **Conhecimento do atacante** — em **caixa-branca**, ele conhece o modelo e seus gradientes; em **caixa-preta**, dispõe apenas de acesso por consulta;
- **Objetivo** — um ataque **não direcionado** busca qualquer erro; um **direcionado**, uma saída errada específica.

> **🛡️ Contexto de defesa** — Em aplicações civis benignas, o deslocamento de distribuição é um **acidente a mitigar**. Em defesa, é uma **tática do oponente**. Um sistema de detecção de alvos (Capítulo 4) enfrentará camuflagem projetada contra ele; um classificador de OSINT (Capítulo 5), conteúdo forjado para enganá-lo; um agente autônomo (Capítulo 7), ambientes manipulados para induzi-lo ao erro. A pergunta operacional nunca é apenas *"qual a acurácia do modelo?"*, mas ***"quão bem ele resiste a alguém que tenta derrotá-lo?"*** — e a segunda não se responde com o conjunto de teste habitual.

## 8.2 Ataques de envenenamento de dados

Os ataques de **envenenamento** (*data poisoning*) agem no momento mais sensível: o **treino**. Duas variantes importam. Os ataques de **disponibilidade** degradam o desempenho geral, injetando exemplos ruidosos ou mal rotulados. Mais insidiosos são os ataques de **porta dos fundos** (*backdoor*, ou *trojan*): implantam um **gatilho oculto** de modo que o modelo se comporte normalmente na imensa maioria dos casos — passando pela validação usual sem levantar suspeita — e só se comporte mal quando o gatilho específico está presente. **Um modelo com *backdoor* é uma bomba-relógio: parece perfeito até o momento, escolhido pelo adversário, em que falha.**

> **📝 Nota** — O envenenamento tem uma dimensão de **cadeia de suprimentos**. Modelos pré-treinados baixados de repositórios abertos, dados coletados sem curadoria e rotulagem terceirizada são todos vetores de ataque: qualquer um pode carregar um *backdoor* ou vieses implantados. É a mesma preocupação de **procedência** do Capítulo 6, agora como questão de segurança concreta. A defesa começa antes do algoritmo: na **procedência e na curadoria verificáveis** dos dados e dos modelos de que se parte.

O Exercício 8.5 constrói um *backdoor* completo. Por ora, a célula abaixo mostra o essencial da ameaça: um gatilho de $2 \times 2$ pixels, associado no treino a uma classe-alvo, deixa a acurácia limpa **intacta** — e, mesmo assim, sequestra qualquer imagem que o contenha:

In [ ]:
# Backdoor: um gatilho 2x2 no canto sequestra a predicao, sem ferir a
# acuracia limpa -- por isso a validacao usual NAO o detecta.
CLASSE_ALVO = 0

def poe_gatilho(X):
    Xg = X.copy()
    Xg[:, 0, 0:2, 0:2] = 1.0      # quadrado brilhante no canto superior
    return Xg

# envenena 8% do treino: imagens COM gatilho, rotuladas como a classe-alvo
rng = np.random.default_rng(SEMENTE)
n_veneno = int(0.08 * len(X_tr))
idx_v = rng.choice(len(X_tr), n_veneno, replace=False)
X_env = np.concatenate([X_tr, poe_gatilho(X_tr[idx_v])])
y_env = np.concatenate([y_tr, np.full(n_veneno, CLASSE_ALVO, dtype=np.int64)])

torch.manual_seed(SEMENTE)
modelo_bd = treina(CNN(), X_env, y_env)

# teste limpo (parece saudavel) vs. teste com gatilho (sequestrado)
nao_alvo = y_te != CLASSE_ALVO
acc_limpa = acuracia(modelo_bd, X_te, y_te)
X_gatilho = poe_gatilho(X_te[nao_alvo])
with torch.no_grad():
    pred_g = modelo_bd(torch.tensor(X_gatilho,
                                    device=DISPOSITIVO)).argmax(1).cpu().numpy()
taxa_sequestro = (pred_g == CLASSE_ALVO).mean()

print(f"acuracia LIMPA do modelo envenenado: {acc_limpa:.3f}  (parece OK!)")
print(f"taxa de sequestro COM o gatilho:     {taxa_sequestro:.3f}  "
      f"(-> classe {CLASSE_ALVO})")
print("O modelo esconde o backdoor: normal ate o gatilho aparecer.")

**Observe:** o modelo envenenado tem acurácia limpa **idêntica** à de um modelo saudável — passaria em qualquer validação de rotina — e, ainda assim, classifica como a classe-alvo quase toda imagem que carrega o gatilho de 2×2. É a **bomba-relógio** em ação: nenhum teste no conjunto limpo a revela.

---
## 8.3 Exemplos adversariais: FGSM e PGD

A vulnerabilidade mais célebre da era do *deep learning* é o **exemplo adversarial**: uma perturbação **imperceptível**, somada a uma entrada, que inverte a predição de um modelo de alta acurácia **com elevada confiança**. O exemplo canônico — a imagem de um panda que, após uma perturbação invisível, é classificada como gibão — tornou-se o emblema da fragilidade dos modelos.

### 8.3.1 Formulação e o método FGSM

O problema do atacante é encontrar uma perturbação $\delta$, pequena o bastante para passar despercebida ($\lVert \delta \rVert \leq \varepsilon$), que **maximize a perda** do modelo:

$$\max_{\lVert \delta \rVert \leq \varepsilon} \mathcal{L}(\theta, x + \delta, y).$$

O **FGSM** (*Fast Gradient Sign Method*) o resolve em **um único passo**, na direção do sinal do gradiente da perda em relação à entrada:

$$\delta = \varepsilon \cdot \mathrm{sign}\!\left(\nabla_x \mathcal{L}(\theta, x, y)\right), \qquad x_{\mathrm{adv}} = x + \delta.$$

A ideia é a mesma da retropropagação (Capítulo 3), **com o alvo invertido**: em vez de ajustar os *pesos* para reduzir a perda, ajusta-se a *entrada* para aumentá-la. A **Listagem 8.1** implementa o ataque:

In [ ]:
# Listagem 8.1 - Ataque adversarial FGSM em PyTorch.
def fgsm(modelo, x, y, epsilon):
    # Fast Gradient Sign Method: um passo na direcao que AUMENTA a perda.
    x = x.clone().detach().requires_grad_(True)
    perda = F.cross_entropy(modelo(x), y)
    modelo.zero_grad()
    perda.backward()                     # gradiente da perda em relacao a x
    # perturbacao de norma-infinito limitada por epsilon
    x_adv = x + epsilon * x.grad.sign()
    return torch.clamp(x_adv, 0.0, 1.0).detach()   # mantem imagem valida

# Uma perturbacao imperceptivel (epsilon pequeno) pode inverter a
# predicao de um modelo de alta acuracia -- eis a fragilidade central.
x0 = torch.tensor(X_te[:16], device=DISPOSITIVO)
y0 = torch.tensor(y_te[:16], device=DISPOSITIVO)
x_adv = fgsm(modelo, x0, y0, epsilon=0.10)
pred_limpa = modelo(x0).argmax(1)
pred_adv = modelo(x_adv).argmax(1)
print(f"acertos ANTES do ataque:  {(pred_limpa == y0).sum().item()}/16")
print(f"acertos DEPOIS do ataque: {(pred_adv == y0).sum().item()}/16")

In [ ]:
# A perturbacao que engana: limpo | perturbacao (amplificada) | adversarial
i = 0
limpo = x0[i, 0].cpu().numpy()
adv = x_adv[i, 0].cpu().numpy()
delta = adv - limpo

fig, eixos = plt.subplots(1, 3, figsize=(10, 3.4))
for eixo, img, titulo, cmap in [
        (eixos[0], limpo, f"limpo -> {NOMES[pred_limpa[i]]}", "gray"),
        (eixos[1], delta, "perturbacao (x5, centrada)", "RdBu_r"),
        (eixos[2], adv, f"adversarial -> {NOMES[pred_adv[i]]}", "gray")]:
    dados = 0.5 + 5 * img if "perturbacao" in titulo else img
    eixo.imshow(dados, cmap=cmap, vmin=0, vmax=1)
    eixo.set_title(titulo); eixo.axis("off")
plt.suptitle(f"FGSM: a mesma imagem, o rotulo invertido (epsilon=0.10)")
plt.tight_layout(); plt.show()
print(f"norma-inf da perturbacao: {np.abs(delta).max():.3f} "
      f"(<= epsilon = 0.10, por construcao)")

**Observe:** a perturbação (painel central, amplificada 5× para ficar visível) é **imperceptível** na imagem final — e, no entanto, muda o rótulo que o modelo atribui. O olho humano continua vendo a mesma assinatura; o classificador, não.

### 8.3.2 PGD e os ataques físicos

O FGSM é **rápido, mas fraco**. O **PGD** (*Projected Gradient Descent*) o fortalece **iterando** pequenos passos e **projetando**, a cada um, a entrada de volta à bola de raio $\varepsilon$ em torno do original:

$$x^{t+1} = \Pi_\varepsilon\!\left(x^t + \alpha \cdot \mathrm{sign}\!\left(\nabla_x \mathcal{L}(\theta, x^t, y)\right)\right),$$

onde $\Pi_\varepsilon$ denota a projeção na bola de raio $\varepsilon$. O PGD é o **ataque de referência** — uma defesa que resiste a ele é considerada séria; uma que só resiste ao FGSM, não. A **Listagem 8.2** o implementa:

In [ ]:
# Listagem 8.2 - Ataque adversarial PGD (iterado e projetado).
def pgd(modelo, x, y, epsilon, alfa=0.01, passos=40):
    # PGD: FGSM iterado, projetado na bola de raio epsilon em torno de x.
    x_orig = x.clone().detach()
    # inicio aleatorio dentro da bola epsilon
    x_adv = x + torch.empty_like(x).uniform_(-epsilon, epsilon)
    x_adv = torch.clamp(x_adv, 0.0, 1.0).detach()

    for _ in range(passos):
        x_adv = x_adv.requires_grad_(True)
        perda = F.cross_entropy(modelo(x_adv), y)
        modelo.zero_grad()
        perda.backward()
        # passo na direcao do gradiente...
        x_adv = x_adv + alfa * x_adv.grad.sign()
        # ...seguido da projecao de volta a bola de raio epsilon
        delta = torch.clamp(x_adv - x_orig, -epsilon, epsilon)
        x_adv = torch.clamp(x_orig + delta, 0.0, 1.0).detach()
    return x_adv

# FGSM vs. PGD no MESMO orcamento epsilon: o iterado e muito mais forte.
todo_X = torch.tensor(X_te, device=DISPOSITIVO)
todo_y = torch.tensor(y_te, device=DISPOSITIVO)
for eps in [0.05, 0.10]:
    x_f = fgsm(modelo, todo_X, todo_y, eps)
    x_p = pgd(modelo, todo_X, todo_y, eps, passos=20)
    acc_f = (modelo(x_f).argmax(1) == todo_y).float().mean().item()
    acc_p = (modelo(x_p).argmax(1) == todo_y).float().mean().item()
    print(f"epsilon={eps:.2f} | acuracia sob FGSM={acc_f:.3f}  "
          f"sob PGD={acc_p:.3f}")

**Observe:** para o **mesmo orçamento** $\varepsilon$, o PGD derruba a acurácia muito mais do que o FGSM — porque procura o pior caso dentro da bola de perturbação, em vez de dar um único passo. É por isso que **avaliar defesas só contra o FGSM é enganoso**: o adversário real usará o ataque mais forte que puder (Exercício 8.2).

Ainda mais preocupante em defesa: esses ataques **sobrevivem ao mundo físico**. Um **adesivo adversarial** (*adversarial patch*) — um padrão impresso e afixado a um objeto — pode fazer um detector deixar de enxergá-lo ou classificá-lo erroneamente (Capítulo 4). **O adversário não precisa de acesso ao sistema: basta alterar o mundo que o sensor observa.**

> **🛡️ Contexto de defesa** — O ataque físico transforma a robustez adversarial de curiosidade acadêmica em **requisito operacional**. Um sistema de detecção de alvos que se deixa cegar por um padrão impresso é uma vulnerabilidade explorável em campo — uma forma de **camuflagem dirigida ao classificador**, não ao olho humano. Avaliar a resistência de um modelo de visão a perturbações adversariais, físicas e digitais, é parte inseparável de prepará-lo para um **ambiente contestado**.

---
## 8.4 Defesas algorítmicas

Contra esses ataques, um repertório de defesas — **nenhuma completa**.

O **treinamento adversarial** é a defesa mais eficaz conhecida. Sua ideia é treinar o modelo sobre os **próprios exemplos adversariais**, resolvendo um problema de otimização **min-max**: minimizar a perda sobre o **pior caso** dentro da bola de perturbação,

$$\min_\theta\, \mathbb{E}_{(x,y)}\!\left[\max_{\lVert \delta \rVert \leq \varepsilon} \mathcal{L}(\theta, x + \delta, y)\right].$$

A cada lote, geram-se exemplos adversariais (o máximo interno, via PGD) e treina-se o modelo sobre eles (o mínimo externo). O resultado é mais robusto — **ao custo de acurácia em dados limpos e de um treino várias vezes mais caro**. A **Listagem 8.3** implementa o laço:

In [ ]:
# Listagem 8.3 - Treinamento adversarial: a defesa min-max com PGD.
def treina_adversarial(modelo, X_tr, y_tr, epsilon=0.10, epocas=8,
                       lote=128, semente=SEMENTE):
    # Minimiza a perda sobre o PIOR caso dentro da bola epsilon
    # (problema min-max). Reusa pgd() da Listagem 8.2.
    modelo.to(DISPOSITIVO).train()
    otim = torch.optim.Adam(modelo.parameters(), lr=1e-3)
    Xt = torch.tensor(X_tr, device=DISPOSITIVO)
    yt = torch.tensor(y_tr, device=DISPOSITIVO)
    rng = np.random.default_rng(semente)
    for _ in range(epocas):
        for i in rng.permutation(len(Xt))[:len(Xt) // lote * lote].reshape(-1, lote):
            idx = torch.tensor(i, device=DISPOSITIVO)
            x, yb = Xt[idx], yt[idx]
            # 1. maximo interno: exemplos adversariais para o lote atual
            x_adv = pgd(modelo, x, yb, epsilon, alfa=0.02, passos=7)
            # 2. minimo externo: treina o modelo SOBRE os adversariais
            perda = F.cross_entropy(modelo(x_adv), yb)
            otim.zero_grad(); perda.backward(); otim.step()
    return modelo

torch.manual_seed(SEMENTE)
modelo_robusto = treina_adversarial(CNN(), X_tr, y_tr, epsilon=0.10)
acc_r = acuracia(modelo_robusto, X_te, y_te)
acc_c = acuracia(modelo, X_te, y_te)
print(f"acuracia limpa | comum: {acc_c:.3f} | robusto: {acc_r:.3f}")
print(f"custo em acuracia limpa: {acc_r - acc_c:+.3f}")
print("(neste problema facil o custo e' pequeno; a Secao 8.6 mede o que")
print(" de fato importa -- quanto cada modelo AGUENTA sob ataque.)")

**Observe:** o **treinamento adversarial** normalmente cobra um preço em acurácia limpa — o famoso *"não há almoço grátis"*. Aqui, num problema didático fácil, esse custo é quase nulo; em problemas reais (mais classes, sinais mais sutis, orçamentos $\varepsilon$ maiores) ele é significativo. A questão nunca é *se* há um custo, mas se ele **vale a pena** frente à ameaça real — e a curva da Seção 8.6 torna essa troca **mensurável**, medindo o que de fato importa: quanto cada modelo **aguenta sob ataque**.

Duas outras linhas completam o quadro. A **destilação defensiva** treina um segundo modelo sobre as saídas suavizadas (*soft labels*) do primeiro, alisando a superfície de decisão e reduzindo a magnitude dos gradientes — o que dificulta o ataque; ataques mais fortes, contudo, a contornam parcialmente. A **detecção**, em vez de tornar o modelo robusto, busca **identificar e rejeitar** entradas adversariais.

> **⚠️ Armadilha comum** — **Declarar um modelo "robusto" sem qualificar a ameaça.** Robustez não é uma propriedade absoluta: só faz sentido em relação a um **modelo de ameaça explícito** — que tipo de ataque, sob que norma, com que orçamento $\varepsilon$. Uma defesa que resiste ao FGSM pode ruir sob PGD; uma que resiste a perturbações de norma $\ell_\infty$ pode falhar em outra norma. Pior, a história do campo é uma **sucessão de defesas anunciadas como seguras e depois quebradas**. A postura honesta é tratar a robustez como um **espectro medido contra ameaças declaradas**, jamais como uma garantia — e desconfiar de qualquer alegação de segurança que não especifique **contra o quê**.

## 8.5 Verificação e validação em sistemas críticos

Chegamos à disciplina que fecha o Módulo III e abre o Módulo IV. Em um sistema crítico, **"funcionou nos testes" é insuficiente**. A **verificação e validação (V&V)** é o processo disciplinado de estabelecer **confiança justificada** em um modelo antes de confiá-lo a uma operação. Ela recolhe, e formaliza, o fio metodológico de todo o livro. Seus elementos são cumulativos:

- **Avaliação rigorosa** — além da acurácia média, examina a **cauda** (pior caso, taxa de falha), como no Capítulo 7;
- **Avaliação de robustez** — mede o desempenho sob o **modelo de ameaça** relevante (FGSM, PGD, ataques físicos), não apenas em dados limpos;
- ***Red teaming*** — submete o modelo a uma equipe dedicada a derrotá-lo, expondo falhas que o teste passivo não revela;
- **Monitoramento em produção** — acompanha o deslocamento de distribuição e a degradação ao longo do tempo;
- **Documentação e rastreabilidade** — sementes, versões, dados, decisões — tornam o sistema **auditável**;
- **Envelope operacional** — as condições sob as quais o modelo foi validado e **fora das quais ele não deve ser empregado**.

Para componentes especialmente críticos, a **verificação formal** busca *provar* propriedades do comportamento do modelo — por exemplo, que a predição não muda para nenhuma perturbação abaixo de um limiar (**robustez certificada**). É difícil e escala mal, mas oferece o que a avaliação empírica não pode: uma **garantia**, ainda que restrita a perturbações pequenas.

> **📝 Nota** — A V&V é o que transforma um modelo promissor em uma **capacidade operacional confiável**. Não é uma etapa final de carimbo, mas um **processo contínuo** que acompanha o modelo do desenvolvimento ao descarte. E é a fundação técnica sobre a qual repousa a governança do Capítulo 10: *não se pode exigir responsabilidade por um sistema que não se sabe verificar, nem manter controle humano significativo sobre um modelo cujo comportamento não se compreende nem se testa*.

---
## 8.6 Aplicação integrada: avaliação de robustez de um modelo

A aplicação que fecha o módulo é uma **avaliação de robustez** — um exercício de *red team* sobre um modelo já treinado. O procedimento é sistemático: mede-se a acurácia limpa e, em seguida, a acurácia sob ataque PGD em **intensidades crescentes de $\varepsilon$**, produzindo uma **curva de robustez**. A **Listagem 8.4** implementa esse núcleo — e o aplicamos **lado a lado** aos dois modelos: o comum e o adversarialmente treinado.

In [ ]:
# Listagem 8.4 - Avaliacao de robustez: curva de acuracia sob ataque.
def curva_robustez(modelo, X, y, epsilons, lote=256):
    # Red team: acuracia sob ataque PGD em funcao da forca epsilon.
    # O nucleo de uma avaliacao honesta de fragilidade. Reusa pgd().
    modelo.eval()
    Xt = torch.tensor(X, device=DISPOSITIVO)
    yt = torch.tensor(y, device=DISPOSITIVO)
    acuracias = []
    for eps in epsilons:
        acertos = 0
        for i in range(0, len(Xt), lote):
            xb, yb = Xt[i:i + lote], yt[i:i + lote]
            x_aval = xb if eps == 0.0 else pgd(modelo, xb, yb, eps, passos=20)
            with torch.no_grad():
                acertos += (modelo(x_aval).argmax(1) == yb).sum().item()
        acuracias.append(acertos / len(yt))
    return acuracias

epsilons = [0.0, 0.01, 0.03, 0.05, 0.10, 0.15]
acc_comum = curva_robustez(modelo, X_te, y_te, epsilons)
acc_robusto = curva_robustez(modelo_robusto, X_te, y_te, epsilons)

print(f"{'epsilon':>8s} | {'modelo comum':>13s} | {'modelo robusto':>15s}")
for e, ac, ar in zip(epsilons, acc_comum, acc_robusto):
    print(f"{e:8.2f} | {ac:13.3f} | {ar:15.3f}")

In [ ]:
# A curva de robustez: onde o modelo de laboratorio DESABA
plt.figure(figsize=(8, 5))
plt.plot(epsilons, acc_comum, "o-", lw=2, label="modelo comum")
plt.plot(epsilons, acc_robusto, "s-", lw=2, label="treinamento adversarial")
plt.axhline(0.25, color="gray", ls=":", label="acaso (4 classes)")
plt.xlabel("forca do ataque (epsilon, norma-inf)")
plt.ylabel("acuracia sob ataque PGD")
plt.title("Curva de robustez: acuracia limpa nao conta a historia toda")
plt.legend(); plt.ylim(-0.02, 1.02)
plt.tight_layout(); plt.show()

# indicador operacional: menor epsilon que derruba a acuracia a metade
def epsilon_meia_vida(epsilons, acc):
    limite = acc[0] / 2
    for e, a in zip(epsilons, acc):
        if a < limite:
            return e
    return None
print(f"epsilon que reduz a acuracia a METADE:")
print(f"  modelo comum:   {epsilon_meia_vida(epsilons, acc_comum)}")
print(f"  modelo robusto: {epsilon_meia_vida(epsilons, acc_robusto)}")

**Observe:** os dois modelos são **empatados em $\varepsilon = 0$** (a acurácia limpa que a validação de rotina reportaria) — e, no entanto, comportam-se de forma **radicalmente diferente sob ataque**: o modelo comum **desaba** sob perturbação pequena, enquanto o adversarialmente treinado **degrada com graça**, sustentando acurácia útil onde o outro já colapsou. É exatamente a informação que a acurácia limpa **esconde** — e que uma decisão de emprego precisa.

O produto dessa avaliação **não é um número, mas um juízo**: a curva de robustez, o **envelope operacional** em que o modelo é confiável e a recomendação — que pode, e por vezes deve, ser a de **não empregar** um modelo que desaba sob perturbação mínima, por mais alta que seja sua acurácia limpa. É a aplicação mais direta da tese do livro: em defesa, **o desempenho em condições benignas é uma promessa vazia se o modelo não resiste ao adversário que certamente o enfrentará**. A honestidade sobre a fragilidade é o começo da segurança; a decisão de empregar, ou não, permanece **humana e informada**.

---
## 8.7 O caminho à frente

Com a robustez adversarial e a V&V, **encerra-se o Módulo III** — e com ele a decisão: a capacidade de agir e a disciplina de tornar essa ação confiável diante de um oponente. Resta transformar tudo o que se construiu em **capacidade operacional e governada**. É o **Módulo IV**.

O **Capítulo 9** trata da **IA embarcada e da computação em tempo real**: as restrições reais de energia, memória e latência das plataformas de campo, a operação em ambientes negados de conectividade e as técnicas — quantização, *pruning*, destilação — que fazem um modelo **caber e correr onde precisa**. O **Capítulo 10** fecha a obra com a **ética, a regulação e os sistemas de armas autônomos**: o marco normativo, o princípio do **controle humano significativo** e a responsabilidade.

A V&V deste capítulo é a **ponte**: fornece a base técnica — modelos avaliados, robustos dentro de um envelope, auditáveis — sobre a qual a confiabilidade do Módulo IV e a governança do Capítulo 10 podem, enfim, se apoiar.

## 📋 Resumo do capítulo

- Modelos de aprendizado são **intrinsecamente vulneráveis**: aprendem correlações, não conceitos robustos, e suas fronteiras de decisão passam perto de exemplos corretos. Em defesa, o adversário é **real e adaptativo**, e desloca a distribuição de propósito.

- Os ataques organizam-se pelo **momento** (envenenamento no treino vs. evasão na inferência) e pelo **conhecimento** do atacante (caixa-branca vs. caixa-preta). O envenenamento inclui ***backdoors*** — gatilhos ocultos que passam pela validação usual.

- Os **exemplos adversariais** invertem predições com perturbações imperceptíveis: o **FGSM** dá um passo no sinal do gradiente; o **PGD** o itera e projeta, sendo o **ataque de referência**. Tais ataques **sobrevivem ao mundo físico** (*patches* adversariais).

- As defesas — **treinamento adversarial** (min-max, a mais eficaz), destilação defensiva e detecção — reduzem, mas **não eliminam** a vulnerabilidade. Nenhuma é completa; robustez só faz sentido contra um **modelo de ameaça explícito**.

- A **verificação e validação** estabelece confiança justificada: avaliação da cauda e da robustez, *red teaming*, monitoramento em produção, rastreabilidade e, sobretudo, um **envelope operacional** definido; a verificação formal oferece garantias restritas.

- A avaliação de robustez produz um **juízo, não um número**: um modelo que desaba sob perturbação mínima **não deve ser empregado**, por alta que seja sua acurácia limpa. A V&V é a **fundação técnica** da governança do Capítulo 10.

## ⚠️ Armadilhas comuns

1. **Confundir acurácia limpa com robustez.** Um modelo excelente em dados benignos pode ruir sob perturbação mínima. Avalie sempre sob o modelo de ameaça relevante, não apenas no conjunto de teste habitual.

2. **Declarar robustez sem qualificar a ameaça.** Robustez só existe em relação a um tipo de ataque, uma norma e um orçamento $\varepsilon$ explícitos. Uma defesa contra FGSM pode falhar sob PGD.

3. **Ignorar a procedência de dados e modelos.** Modelos pré-treinados e dados não curados podem carregar *backdoors* ou vieses implantados. A defesa começa na curadoria e na verificação da cadeia de suprimentos.

4. **Confiar em uma defesa por ela ainda não ter sido quebrada.** O campo é uma corrida; defesas anunciadas como seguras foram sucessivamente contornadas. Trate a robustez como espectro medido, não como garantia.

5. **Subestimar os ataques físicos.** Perturbações adversariais sobrevivem à impressão e à captura por sensor. Um detector de visão para ambiente contestado deve ser avaliado contra camuflagem dirigida ao classificador.

6. **Empregar sem V&V e sem envelope operacional.** "Funcionou nos testes" não basta em sistema crítico. Defina o envelope de validade, monitore em produção e mantenha a decisão de empregar **humana e informada** (Capítulo 10).

---
## 🧭 Exercícios

Classificação: **Essencial** (fixação) · **Tático** (aplicação) · **Estratégico** (extensão criativa). Os exercícios de código têm células preparadas abaixo; os dissertativos podem ser respondidos em células de texto no próprio notebook.

### Essencial

**Exercício 8.1** — Aplique o FGSM (Listagem 8.1) ao modelo treinado com $\varepsilon \in \{0{,}01;\ 0{,}03;\ 0{,}10\}$. Relate a **queda de acurácia** em cada nível e verifique, visualmente, que as perturbações menores são **imperceptíveis**. Explique, em duas linhas, por que o ataque usa o **sinal** do gradiente.

In [ ]:
# Exercicio 8.1 - a queda de acuracia em funcao de epsilon (FGSM)
Xt = torch.tensor(X_te, device=DISPOSITIVO)
yt = torch.tensor(y_te, device=DISPOSITIVO)

print(f"acuracia limpa: {(modelo(Xt).argmax(1) == yt).float().mean():.3f}\n")
for eps in [0.01, 0.03, 0.10]:     # <--- ALTERE e explore outros valores
    x_adv = fgsm(modelo, Xt, yt, eps)
    acc = (modelo(x_adv).argmax(1) == yt).float().mean().item()
    print(f"epsilon = {eps:.2f} -> acuracia sob FGSM = {acc:.3f}")

# visualiza a imperceptibilidade em epsilon=0.03
x1 = fgsm(modelo, Xt[:1], yt[:1], 0.03)
fig, e = plt.subplots(1, 2, figsize=(5.5, 3))
e[0].imshow(Xt[0, 0].cpu(), cmap="gray"); e[0].set_title("limpo"); e[0].axis("off")
e[1].imshow(x1[0, 0].cpu(), cmap="gray"); e[1].set_title("adversarial"); e[1].axis("off")
plt.tight_layout(); plt.show()

# Sua resposta (2 linhas):
# O ataque usa o SINAL do gradiente (e nao o gradiente cheio) porque ...

**Exercício 8.2** — A célula abaixo compara, sobre o mesmo modelo e o mesmo $\varepsilon$, a eficácia do **FGSM** (Listagem 8.1) e do **PGD** (Listagem 8.2). Execute-a e explique, em poucas linhas, por que o PGD é um ataque **mais forte** e por que ele é adotado como **referência** para avaliar defesas.

In [ ]:
# Exercicio 8.2 - FGSM vs. PGD no mesmo orcamento
for eps in [0.03, 0.05, 0.10]:     # <--- ALTERE e observe a diferenca crescer
    x_f = fgsm(modelo, Xt, yt, eps)
    x_p = pgd(modelo, Xt, yt, eps, passos=20)
    acc_f = (modelo(x_f).argmax(1) == yt).float().mean().item()
    acc_p = (modelo(x_p).argmax(1) == yt).float().mean().item()
    print(f"epsilon={eps:.2f} | FGSM deixa {acc_f:.3f} | "
          f"PGD deixa {acc_p:.3f} de acuracia")

# Sua resposta (poucas linhas):
# 1) O PGD e mais forte porque, em vez de UM passo, ele ...
# 2) E' a referencia para avaliar defesas porque uma defesa que resiste
#    ao ataque mais forte conhecido ...

**Exercício 8.3** — Com a Listagem 8.4, trace a curva de robustez do **modelo sem defesa** (já calculada em `acc_comum`). Identifique o **menor $\varepsilon$ que reduz a acurácia à metade** e discuta, em duas linhas, o que essa curva revela sobre a adequação do modelo a um ambiente adversarial.

In [ ]:
# Exercicio 8.3 - o epsilon de "meia-vida" da acuracia
# (reusa acc_comum e epsilons da Secao 8.6)
limite = acc_comum[0] / 2
print(f"acuracia limpa: {acc_comum[0]:.3f} | metade: {limite:.3f}\n")
for e, a in zip(epsilons, acc_comum):
    marca = "  <-- primeiro abaixo da metade" if a < limite else ""
    print(f"epsilon={e:.2f}  acuracia={a:.3f}{marca}")

# Sua resposta (2 linhas):
# 1) O menor epsilon que corta a acuracia pela metade e' ...
# 2) Isso revela que, num ambiente adversarial, o modelo ...

### Tático

**Exercício 8.4** — A célula abaixo compara, num mesmo gráfico, as curvas de robustez do modelo **comum** e do **adversarialmente treinado** (já calculadas). Quantifique tanto o **ganho de robustez** quanto a **perda de acurácia limpa**, e discuta a troca à luz de um requisito operacional concreto (por exemplo: *"o sistema precisa manter ≥ 70% de acurácia sob $\varepsilon = 0{,}05$"*).

In [ ]:
# Exercicio 8.4 - quantificando a troca robustez vs. acuracia limpa
print(f"{'epsilon':>8s} | {'comum':>7s} | {'robusto':>8s} | {'ganho':>7s}")
for e, ac, ar in zip(epsilons, acc_comum, acc_robusto):
    print(f"{e:8.2f} | {ac:7.3f} | {ar:8.3f} | {ar - ac:+7.3f}")

print(f"\ncusto em acuracia LIMPA (epsilon=0): "
      f"{acc_robusto[0] - acc_comum[0]:+.3f}  "
      f"(pequeno AQUI; real em problemas dificeis)")
REQUISITO_EPS, REQUISITO_ACC = 0.05, 0.70   # <--- ALTERE o requisito
for nome, curva in [("comum", acc_comum), ("robusto", acc_robusto)]:
    a = curva[epsilons.index(REQUISITO_EPS)]
    ok = "ATENDE" if a >= REQUISITO_ACC else "NAO atende"
    print(f"requisito (acc>={REQUISITO_ACC} sob eps={REQUISITO_EPS}): "
          f"modelo {nome} {ok} (tem {a:.3f})")

# Sua resposta (poucas linhas):
# A troca vale a pena SE o ambiente de emprego for adversarial, porque ...

**Exercício 8.5** — A célula abaixo reproduz o **ataque de *backdoor*** da Seção 8.2 com a fração de envenenamento ajustável. Execute-a e mostre que o modelo mantém **alta acurácia no teste limpo** mas classifica como a classe-alvo qualquer entrada com o gatilho. Discuta por que a **validação usual não detecta** o ataque, e experimente reduzir `fracao_veneno` para achar o mínimo que ainda instala o *backdoor*.

In [ ]:
# Exercicio 8.5 - o backdoor e o minimo de veneno que o instala
def treina_backdoor(fracao_veneno, classe_alvo=0):
    rng = np.random.default_rng(SEMENTE)
    n_v = int(fracao_veneno * len(X_tr))
    idx = rng.choice(len(X_tr), n_v, replace=False)
    Xe = np.concatenate([X_tr, poe_gatilho(X_tr[idx])])
    ye = np.concatenate([y_tr, np.full(n_v, classe_alvo, dtype=np.int64)])
    torch.manual_seed(SEMENTE)
    m = treina(CNN(), Xe, ye)
    nao_alvo = y_te != classe_alvo
    Xg = poe_gatilho(X_te[nao_alvo])
    with torch.no_grad():
        seq = (m(torch.tensor(Xg, device=DISPOSITIVO)).argmax(1).cpu().numpy()
               == classe_alvo).mean()
    return acuracia(m, X_te, y_te), seq

for fracao in [0.005, 0.02, 0.08]:   # <--- ALTERE: qual o minimo eficaz?
    acc, seq = treina_backdoor(fracao)
    print(f"veneno={fracao:.1%} | acuracia limpa={acc:.3f} | "
          f"taxa de sequestro={seq:.3f}")

# Sua resposta (poucas linhas):
# 1) A validacao usual nao detecta porque ela so testa dados LIMPOS, e ...
# 2) O minimo de veneno que ainda instala o backdoor foi cerca de ...

**Exercício 8.6** *(dissertativo)* — Considere um sistema de **detecção de alvos** (Capítulo 4) a ser empregado em ambiente contestado. Elabore um **plano de *red teaming*** que inclua ataques digitais (PGD) e a avaliação da resistência a um ***patch* adversarial físico**. Especifique as métricas, os modelos de ameaça e o **critério de aprovação** para emprego.

*Responda em uma célula de texto abaixo.*

### Estratégico

**Exercício 8.7** — Em um texto técnico (até três páginas), defenda a tese de que a **robustez adversarial é um requisito operacional**, e não acadêmico, para a IA de defesa. Aborde a corrida entre ataque e defesa, a ausência de garantias absolutas e as implicações disso para a decisão de empregar — ou não — um modelo em ambiente contestado.

**Exercício 8.8** — Projete um **protocolo completo de V&V** para um modelo crítico à sua escolha, especificando: os testes de robustez e o modelo de ameaça, o processo de *red teaming*, o monitoramento em produção, a documentação exigida para auditabilidade e a definição explícita do **envelope operacional**. Justifique como esse protocolo sustenta a governança do Capítulo 10.

**Exercício 8.9** — Escolha um modelo e um conjunto de dados públicos e conduza um **estudo completo de robustez**: avalie o modelo sem defesa, aplique o treinamento adversarial e reavalie, documentando em um miniartigo de até três páginas as **curvas de robustez**, a troca entre robustez e acurácia limpa, os modelos de ameaça considerados e uma recomendação fundamentada de emprego, com seu **envelope de validade**.

---

*Fim do Capítulo 8 — encerramos o Módulo III. No Capítulo 9, o Módulo IV: a IA embarcada e a computação em tempo real, onde o modelo precisa caber e correr na plataforma de campo.*